# 트랜스포머 논문 읽기, 어텐션 한 층을 numpy로 재현

NumPy로 어텐션 한 층을 구현하고 스케일링·마스크·위치 인코딩·계산 비용을 네 실험으로 확인합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/transformer_attention_numpy.ipynb)

[Attention Is All You Need 논문](https://arxiv.org/abs/1706.03762)

Colab 기본 CPU 런타임에서 위에서부터 순서대로 실행하세요. 외부 의존성은 NumPy뿐이며 추가 설치나 데이터 다운로드는 없습니다.
잔차 연결·LayerNorm·FFN·학습은 포함하지 않습니다.


In [ ]:
import platform
import time

import numpy as np

print(f"NumPy: {np.__version__}")
print(f"Platform: {platform.system()} {platform.release()} {platform.machine()}")


## 어텐션과 위치 인코딩 함수

논문 3.2·3.5절의 scaled dot-product attention, 멀티헤드 투영, 인과 마스크, 사인 위치 인코딩을 구현합니다.
마스크의 `True`는 해당 연결을 차단하며, 어텐션 함수는 출력과 가중치를 함께 반환합니다.


In [ ]:
"""논문 3.2·3.5절의 어텐션과 사인 위치 인코딩을 NumPy로 구현한다."""

import numpy as np


def softmax(x, axis=-1):
    """최댓값을 빼 안정화한다. 전부 마스킹된 행은 ValueError로 알린다."""
    x = np.asarray(x)
    dtype = np.result_type(x.dtype, np.float32)
    out = np.array(x, dtype=dtype, copy=True)
    maximum = np.max(out, axis=axis, keepdims=True)
    if np.any(~np.isfinite(maximum)):
        raise ValueError("softmax needs a finite maximum in every row")
    out -= maximum
    np.exp(out, out=out)
    out /= np.sum(out, axis=axis, keepdims=True)
    return out


def scaled_dot_product_attention(Q, K, V, mask=None, scale=True):
    """(..., queries, d_k) 입력을 받아 (출력, 가중치)를 반환한다.

    mask는 점수 배열에 브로드캐스트 가능해야 하며 True인 연결을 차단한다.
    """
    Q, K, V = map(np.asarray, (Q, K, V))
    if Q.shape[-1] != K.shape[-1] or K.shape[-2] != V.shape[-2]:
        raise ValueError("incompatible Q, K, V dimensions")
    scores = Q @ np.swapaxes(K, -1, -2)
    if scale:
        scores /= np.sqrt(Q.shape[-1])
    if mask is not None:
        scores = np.where(np.asarray(mask, dtype=bool), -np.inf, scores)
    weights = softmax(scores)
    return weights @ V, weights


def causal_mask(n):
    """현재 위치보다 미래인 열만 True인 (n, n) 마스크."""
    return np.triu(np.ones((n, n), dtype=bool), k=1)


def sinusoidal_pe(n, d_model):
    """PE(pos, 2i)=sin(pos*ω_i), PE(pos, 2i+1)=cos(pos*ω_i)."""
    positions = np.arange(n, dtype=np.float64)[:, None]
    frequencies = 10000.0 ** (-np.arange(0, d_model, 2) / d_model)
    angles = positions * frequencies
    pe = np.empty((n, d_model), dtype=np.float64)
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles[:, :d_model // 2])
    return pe


def self_attention_layer(x, Wq, Wk, Wv, Wo, n_heads, mask=None):
    """멀티헤드 투영·어텐션·출력 투영 결과 (n, d_model)를 반환한다.

    잔차 연결, LayerNorm, FFN은 포함하지 않는다. 가중치 행렬은 (d, d).
    """
    n, d_model = x.shape
    if n_heads <= 0 or d_model % n_heads:
        raise ValueError("d_model must be divisible by positive n_heads")
    d_k = d_model // n_heads
    Q, K, V = [
        (x @ W).reshape(n, n_heads, d_k).transpose(1, 0, 2)
        for W in (Wq, Wk, Wv)
    ]
    heads, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
    return heads.transpose(1, 0, 2).reshape(n, d_model) @ Wo


## 실험 1. √d_k 스케일링

차원 `d_k`가 커질 때 내적 점수의 분산과 softmax 포화가 어떻게 바뀌는지 확인합니다.
스케일링 전후의 최대 확률, 로짓에 대한 softmax 야코비안의 Frobenius 노름, 최대 확률이 0.99 이상인 비율을 비교합니다.


In [ ]:
"""점수 분산·softmax 포화·야코비안 노름으로 √d_k 스케일링을 확인한다."""

import numpy as np



def statistics(logits):
    p = softmax(logits)
    maximum = p.max(axis=1)
    s2 = np.sum(p * p, axis=1)
    s3 = np.sum(p * p * p, axis=1)
    # ||diag(p) - ppᵀ||_F² = Σp² - 2Σp³ + (Σp²)².
    norm = np.sqrt(np.maximum(s2 - 2 * s3 + s2 * s2, 0))
    return maximum.mean(), norm.mean(), np.mean(maximum >= 0.99)


def main():
    rng = np.random.default_rng(2017)
    trials, keys = 2000, 64
    print(f"seed=2017; trials={trials}; keys={keys}; q,k ~ N(0,1)")
    print("Jacobian: derivative with respect to softmax logits (not q or k)")
    print("d_k  sample_var  theory  max_raw max_scaled  J_F_raw J_F_scaled  >=.99_raw >=.99_scaled")
    for d_k in (16, 64, 256, 1024):
        logits = np.empty((trials, keys))
        for start in range(0, trials, 100):
            q = rng.standard_normal((100, d_k))
            k = rng.standard_normal((100, keys, d_k))
            logits[start:start + 100] = np.einsum("bd,bkd->bk", q, k)
        raw, scaled = statistics(logits), statistics(logits / np.sqrt(d_k))
        print(f"{d_k:4d} {logits.var(ddof=1):11.4f} {d_k:7d} "
              f"{raw[0]:8.5f} {scaled[0]:10.5f} {raw[1]:8.5f} {scaled[1]:10.5f} "
              f"{raw[2]:10.4%} {scaled[2]:12.4%}")


main()


## 실험 2. 디코더 마스크

인과 마스크가 미래 토큰으로 향하는 어텐션 가중치를 정확히 0으로 만드는지 확인합니다.
미래 토큰만 바꿨을 때 현재까지의 출력이 유지되는지, 마스크가 없는 경우와 비교합니다.


In [ ]:
"""인과 마스크의 미래 가중치와 미래 토큰 변경에 대한 출력 불변성을 확인한다."""

import numpy as np



def main():
    rng = np.random.default_rng(2018)
    n, d, heads = 8, 16, 1
    x = rng.standard_normal((n, d))
    weights = [rng.standard_normal((d, d)) / np.sqrt(d) for _ in range(4)]
    Wq, Wk, Wv, _ = weights
    _, a = scaled_dot_product_attention(x[:5] @ Wq, x[:5] @ Wk,
                                       x[:5] @ Wv, causal_mask(5))
    print("seed=2018; n=8; d_model=16; heads=1")
    print("(a) n=5 causal attention weights:")
    print(np.array2string(a, precision=2, floatmode="fixed"))
    upper_zero = bool(np.all(a[np.triu_indices(5, k=1)] == 0))
    print(f"upper triangle exactly zero: {upper_zero}")
    assert upper_zero
    baseline = [self_attention_layer(x, *weights, heads, mask=m)
                for m in (causal_mask(n), None)]
    print("(b) t (zero-based)  masked_max_abs_diff  unmasked_max_abs_diff")
    for t in range(n):
        changed = x.copy()
        changed[t + 1:] = rng.standard_normal((n - t - 1, d))
        diffs = []
        for original, mask in zip(baseline, (causal_mask(n), None)):
            out = self_attention_layer(changed, *weights, heads, mask=mask)
            diffs.append(float(np.max(np.abs(out[:t + 1] - original[:t + 1]))))
        assert diffs[0] == 0.0
        print(f"{t:18d} {diffs[0]:20.1f} {diffs[1]:22.12g}")
    print("t=7 has no future tokens to change.")


main()


## 실험 3. 위치 인코딩

위치 인코딩이 없는 어텐션의 순열 등변성과 고정 위치 인코딩을 더했을 때의 변화를 확인합니다.
사인 위치 인코딩의 위치 이동을 회전으로 표현하고, 두 위치의 내적이 상대 거리에 의존하는지도 살펴봅니다.
부동소수점 반올림에 따른 작은 오차는 NumPy·BLAS·플랫폼에 따라 달라질 수 있습니다.


In [ ]:
"""순열 등변성, 위치 이동의 회전 표현, PE 내적의 상대 위치 의존성을 확인한다."""

import numpy as np



def main():
    rng = np.random.default_rng(2019)
    d, heads, n = 512, 8, 12
    weights = [rng.standard_normal((d, d)) / np.sqrt(d) for _ in range(4)]

    def layer(x):
        return self_attention_layer(x, *weights, heads)

    x = rng.standard_normal((n, d))
    permutation = rng.permutation(n)
    pe = sinusoidal_pe(n, d)
    no_pe = np.max(np.abs(layer(x[permutation]) - layer(x)[permutation]))
    with_pe = np.max(np.abs(layer(x[permutation] + pe) - layer(x + pe)[permutation]))
    print("seed=2019; d_model=512; heads=8; float64; no causal mask")
    print(f"(a) max|f(Px)-Pf(x)| without PE: {no_pe:.12g}")
    print(f"    max|f(Px+PE)-Pf(x+PE)| with fixed-position PE: {with_pe:.12g}")
    assert no_pe <= 1e-10
    tokens = rng.standard_normal((3, d))  # dog, bites, man
    first, second = layer(tokens), layer(tokens[::-1])
    print('    "dog bites man" / "man bites dog", without PE:')
    for i, word in enumerate(("dog", "bites", "man")):
        diff = np.max(np.abs(first[i] - second[2 - i]))
        print(f"    {word}: corresponding-word max difference = {diff:.12g}")
        assert diff <= 1e-10

    print("(b) PE(pos+k) from PE(pos), pos=0..999:")
    print("    column pair [sin,cos]: R=[[cos θ,sin θ],[-sin θ,cos θ]]")
    frequencies = 10000.0 ** (-np.arange(0, d, 2) / d)
    base = sinusoidal_pe(1000, d).reshape(1000, d // 2, 2)
    for k in (1, 5, 50):
        theta = k * frequencies
        rotation = np.empty((d // 2, 2, 2))
        rotation[:, 0, 0] = rotation[:, 1, 1] = np.cos(theta)
        rotation[:, 0, 1] = np.sin(theta)
        rotation[:, 1, 0] = -np.sin(theta)
        shifted = np.einsum("hij,phj->phi", rotation, base).reshape(1000, d)
        error = np.max(np.abs(shifted - sinusoidal_pe(1000 + k, d)[k:]))
        assert error <= 1e-9
        print(f"    k={k:2d}: max error = {error:.12g}")

    print("(c) PE(i) dot PE(i+offset), i=0..999 (population std):")
    print("offset         mean              std")
    all_pe = sinusoidal_pe(1256, d)
    for offset in (0, 1, 2, 4, 8, 16, 64, 256):
        dots = np.sum(all_pe[:1000] * all_pe[offset:offset + 1000], axis=1)
        print(f"{offset:6d} {dots.mean():16.9f} {dots.std():16.9g}")


main()


## 실험 4. RNN과 어텐션 비용

같은 입력에서 RNN과 멀티헤드 어텐션의 실행 시간, 행렬곱 FLOPs, 순차 의존 단계를 비교합니다.
시간은 워밍업 1회 뒤 5회 측정한 중앙값·최소·최대이며, 배열 할당을 포함하고 FLOPs는 tanh·softmax·스케일링을 제외합니다.
기본 BLAS 설정으로 잰 값이며 하드웨어에 따라 크게 달라진다. 긴 시퀀스에서는 실행 시간과 메모리 사용량이 커질 수 있습니다.


In [ ]:
def rnn_layer(x, W, U):
    h = np.zeros(x.shape[1], dtype=x.dtype)
    output = np.empty_like(x)
    for t in range(len(x)):
        h = np.tanh(W @ h + U @ x[t])
        output[t] = h
    return output


def measure(fn):
    fn()  # warm-up, excluded
    times = []
    for _ in range(5):
        start = time.perf_counter()
        fn()
        times.append((time.perf_counter() - start) * 1000)
    return np.median(times), min(times), max(times)


def main():
    config = np.show_config(mode="dicts")
    blas_name = config.get("Build Dependencies", {}).get("blas", {}).get("name", "not reported")
    print(f"NumPy: {np.__version__}; BLAS: {blas_name}")
    print(f"Platform: {platform.system()} {platform.release()} {platform.machine()}")
    print("seed=2020; CPU float32; d=512; heads=8")
    print("1 warm-up + 5 timed runs per method and n; allocation included")
    print("FLOPs count matmuls only; excludes tanh, softmax, scaling; multiply-add=2")
    print("Sequential steps: Table 1 dependency abstraction; not CPU instruction count")
    print("    n method       median_ms       min_ms       max_ms        FLOPs  steps      GFLOP/s")
    rng = np.random.default_rng(2020)
    d = 512
    matrices = [rng.standard_normal((d, d), dtype=np.float32) / np.sqrt(np.float32(d))
                for _ in range(6)]
    W, U, *weights = matrices
    for n in (128, 256, 512, 1024, 2048, 4096):
        x = rng.standard_normal((n, d), dtype=np.float32)
        methods = (
            ("RNN", lambda: rnn_layer(x, W, U), 2 * n * d**2 * 2, n),
            ("attention", lambda: self_attention_layer(x, *weights, 8),
             4 * n * d**2 * 2 + 2 * n**2 * d * 2, 1),
        )
        for name, fn, flops, steps in methods:
            median, minimum, maximum = measure(fn)
            print(f"{n:5d} {name:9s} {median:13.3f} {minimum:12.3f} {maximum:12.3f} "
                  f"{flops:12d} {steps:6d} {flops / (median * 1e6):12.3f}", flush=True)


main()
